# Defect Formation Energy

Calculate the neutral defect formation energy (eV) of a defective supercell using a multi-material DFT workflow on the Mat3ra platform.

The workflow takes **two materials, in order**:

- **[0] Defective supercell** — the structure containing the defect(s) (vacancy, substitution, interstitial, or a mix).
- **[1] Pristine supercell** — the defect-free version of the *same* supercell.

Only the defective cell is computed here (`pw_scf`). The pristine reference energy is fetched from a previously-finished **Total Energy** job, so the pristine must already be converged/relaxed and have such a job on the platform. Elemental chemical potentials are taken from Standata elemental reference materials (each needs a total energy too), as in [Formation Energy](formation_energy.ipynb).

Formula:

$$E_{\text{defect}} = E_{\text{defective}}[q] - E_{\text{pristine}} - \sum_i \Delta N_i\, \mu_i + q\,(E_{\text{VBM}} + E_F) \quad [\text{eV}]$$

Set `CHARGE` (cell 1.3) to a non-zero value to charge the defective supercell (`tot_charge` in the QE `&SYSTEM` namelist, compensated by a uniform jellium background). **This notebook does not compute a charged-defect finite-size correction** (e.g. Freysoldt-Neugebauer-Van de Walle) or track the Fermi-level term $q(E_{\text{VBM}}+E_F)$ -- for `CHARGE = 0` (the default) neither is needed and the value below is directly physical; for `CHARGE != 0` the reported value is the raw, uncorrected total-energy difference only.

where $\Delta N_i = \text{count}_i(\text{defective}) - \text{count}_i(\text{pristine})$ is the per-species atom-count change and $\mu_i = E_{\text{elemental},i} / n_{\text{atoms},i}$.

The reported value is the **total** formation energy of the whole defective configuration (per cell), not per defect: for a single isolated defect it equals the point-defect formation energy; for multiple/interacting defects it is the aggregate for that configuration.

<h2 style="color:green">Usage</h2>

1. Build a pristine supercell and a defective version of it (for example via `create_point_defect.ipynb`, which produces both), and export both to `../uploads` — or set the names below to match materials on the platform.
2. Run [Total Energy](total_energy.ipynb) for the pristine supercell, and for each Standata elemental reference material.
3. Set parameters in cells 1.2 and 1.3 below.
4. Click "Run" > "Run All".
5. Inspect the resolved references, their total energies, and the final defect formation energy.

## Summary

1. Set up the environment and parameters.
2. Authenticate and initialize API client.
3. Load defective and pristine supercells, resolve elemental references, verify the pristine total energy, and assemble the job materials.
4. Configure the Defect Formation Energy workflow.
5. Configure compute.
6. Create, submit, and monitor the multi-material job.
7. Retrieve results.

## 1. Set up the environment and parameters
### 1.1. Install packages (JupyterLite)

In [ ]:
from mat3ra.notebooks_utils.packages import install_packages

await install_packages("made|api_examples")

### 1.2. Set parameters

In [ ]:
from datetime import datetime
from mat3ra.ide.compute import QueueName

# 2. Auth and organization parameters
ORGANIZATION_NAME = None

# 3. Material parameters
FOLDER = "../uploads"
DEFECTIVE_NAME = "Si"  # defective supercell: name in uploads or on the platform
PRISTINE_NAME = "Si"  # defect-free version of the SAME supercell

# 4. Workflow parameters
WORKFLOW_SEARCH_TERM = "defect_formation_energy.json"
APPLICATION_NAME = "espresso"
MY_WORKFLOW_NAME = "Defect Formation Energy"

# 5. Compute parameters
CLUSTER_NAME = None  # specify full or partial name i.e. "cluster-001" to select
QUEUE_NAME = QueueName.D
PPN = 1

# 6. Job parameters
timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
POLL_INTERVAL = 30  # seconds

### 1.3. Set specific defect formation energy parameters

In [ ]:
# K-grid for the defective-cell SCF (if not set, KPPRA is used by default)
SCF_KGRID = None  # e.g. [4, 4, 4]

# Net charge on the defective supercell (e.g. -3 for a triply negatively charged
# defect, +1 for a singly positively charged one). 0 = neutral defect (default).
# NOTE: a non-zero charge requires a charged-defect finite-size correction (e.g.
# Freysoldt-Neugebauer-Van de Walle) to remove the spurious electrostatic
# interaction between the charged defect and its periodic images -- this
# notebook does NOT compute that correction, so CHARGE != 0 results are raw,
# uncorrected values only.
CHARGE = 0  # e.g. -3, +1

## 2. Authenticate and initialize API client
### 2.1. Configure API endpoint
Local development: point the API client at a local platform instance. Remove or adjust these for the hosted platform.

### 2.2. Authenticate
Authenticate in the browser and have credentials stored in environment variable `OIDC_ACCESS_TOKEN`.

In [ ]:
from mat3ra.notebooks_utils.auth import authenticate

await authenticate()

### 2.3. Initialize API client

In [ ]:
from mat3ra.api_client import APIClient

client = APIClient.authenticate()
client

### 2.4. Select account to work under

In [ ]:
client.list_accounts()

In [ ]:
selected_account = client.my_account

if ORGANIZATION_NAME:
    selected_account = client.get_account(name=ORGANIZATION_NAME)

ACCOUNT_ID = selected_account.id
print(f"✅ Selected account ID: {ACCOUNT_ID}, name: {selected_account.name}")

### 2.5. Select project

In [ ]:
projects = client.projects.list({"isDefault": True, "owner._id": ACCOUNT_ID})
project_id = projects[0]["_id"]
print(f"✅ Using project: {projects[0]['name']} ({project_id})")

## 3. Load defective and pristine supercells
### 3.1. Load materials from local files or platform

In [ ]:
import re
from mat3ra.made.material import Material
from mat3ra.notebooks_utils.material import load_material_from_folder
from mat3ra.notebooks_utils.ipython.entity.material.visualize import visualize_materials as visualize


def load_material(name):
    loaded = load_material_from_folder(FOLDER, name)
    if loaded is not None:
        print(f"✅ Loaded '{name}' from folder: {loaded.name}")
        return loaded
    matches = client.materials.list({"name": {"$regex": re.escape(name), "$options": "i"}})
    if not matches:
        raise ValueError(f"No material containing '{name}' was found in '{FOLDER}' or on the platform.")
    material = Material.create(matches[0])
    print(f"♻️  Loaded '{name}' from platform: {matches[0]['_id']}")
    return material


defective = load_material(DEFECTIVE_NAME)
pristine = load_material(PRISTINE_NAME)

visualize([{"material": defective, "title": "Defective"},
           {"material": pristine, "title": "Pristine"}], repetitions=[1, 1, 1], rotation="-90x")

### 3.2. Prepare materials for Quantum ESPRESSO
Strip atom labels. Labels are useful for analysis notebooks but are not compatible with QE input generation.

In [ ]:
defective_material = defective.clone()
defective_material.basis.set_labels_from_list([])
pristine_material = pristine.clone()
pristine_material.basis.set_labels_from_list([])
print(f"Prepared defective: {defective_material.name}")
print(f"Prepared pristine:  {pristine_material.name}")

### 3.3. Resolve unique elements (union of defective and pristine)
The chemical-potential term covers every species whose count changes, so elements are taken from the union of the two structures (e.g. a species fully removed by a vacancy still appears via the pristine).

In [ ]:
elements = sorted(
    set(defective_material.basis.elements.values) | set(pristine_material.basis.elements.values)
)
print(f"Elements (defective ∪ pristine): {elements}")

### 3.4. Resolve Standata elemental reference materials
Elemental chemical potentials come from Standata materials tagged `elemental` with `metadata.element`. Each must also have a refined `total_energy` — this is enforced by the workflow at runtime; run [Total Energy](total_energy.ipynb) for any elemental reference that is missing one.

In [ ]:
elemental_materials_data = client.materials.list(
    {"tags": "elemental", "metadata.element": {"$in": elements}},
)
element_materials_by_symbol = {
    el: [m for m in elemental_materials_data if m.get("metadata", {}).get("element") == el]
    for el in elements
}
missing = [el for el in elements if not element_materials_by_symbol.get(el)]
if missing:
    raise RuntimeError(
        f"Missing Standata elemental reference material(s) for {missing}. "
        "Add elemental material(s) tagged 'elemental' with metadata.element before running."
    )
print(f"Resolved elemental reference materials for: {', '.join(elements)}")

### 3.5. Save materials and check the pristine total energy
Save both materials and confirm the pristine already has a finished Total Energy job (its energy is fetched, not recomputed), then assemble the job materials in order: `[0]` defective, `[1]` pristine.

In [ ]:
from mat3ra.notebooks_utils.core.entity.material.api import get_or_create_material
from mat3ra.notebooks_utils.core.entity.property.api import find_total_energy_for_material

saved_defective = Material.create(get_or_create_material(client, defective_material, ACCOUNT_ID))
saved_pristine = Material.create(get_or_create_material(client, pristine_material, ACCOUNT_ID))

_, pristine_te_property = find_total_energy_for_material(client, saved_pristine.id)
if pristine_te_property is None:
    raise RuntimeError("Run total_energy.ipynb for the pristine material first.")

# Order matters: [0] defective (computed), [1] pristine (reference).
materials = [saved_defective, saved_pristine]

## 4. Configure the Defect Formation Energy workflow
### 4.1. Select application

In [ ]:
from mat3ra.ade.application import Application
from mat3ra.standata.applications import ApplicationStandata

app_config = ApplicationStandata.get_by_name_first_match(APPLICATION_NAME)
app = Application(**app_config)
print(f"Using application: {app.name}")

### 4.2. Load workflow from Standata, apply parameters, and preview

In [ ]:
from mat3ra.standata.workflows import WorkflowStandata
from mat3ra.wode.workflows import Workflow
from mat3ra.wode.context.providers import PointsGridDataProvider
from mat3ra.notebooks_utils.ipython.entity.workflow.visualize import visualize_workflow
from mat3ra.notebooks_utils.workflow import patch_workflow_qe_input

defect_workflow_config = WorkflowStandata.filter_by_application(app.name).get_by_name_first_match(
    WORKFLOW_SEARCH_TERM
)
defect_workflow = Workflow.create(defect_workflow_config)
defect_workflow.name = MY_WORKFLOW_NAME
print(f"Loaded workflow: {defect_workflow.name}")
print(f"Multi-material: {getattr(defect_workflow, 'isMultiMaterial', False)}")

# K-grid for the defective-cell SCF.
if SCF_KGRID is not None:
    new_context = PointsGridDataProvider(dimensions=SCF_KGRID, isEdited=True).get_context_item_data()
    for subworkflow in defect_workflow.subworkflows:
        unit = subworkflow.get_unit_by_name(name="pw_scf")
        if unit:
            unit.add_context(new_context)
            subworkflow.set_unit(unit)

# Net charge on the defective-cell SCF: adds `tot_charge` to the &SYSTEM namelist
if CHARGE:
    patch_workflow_qe_input(defect_workflow, {"system": {"tot_charge": CHARGE}}, unit_names=["pw_scf"])

visualize_workflow(defect_workflow)

## 5. Create the compute configuration
### 5.1. Select cluster

In [ ]:
clusters = client.clusters.list()
print(f"Available clusters: {[c['hostname'] for c in clusters]}")

### 5.2. Create compute configuration

In [ ]:
from mat3ra.ide.compute import Compute

if CLUSTER_NAME:
    cluster = next((c for c in clusters if CLUSTER_NAME in c["hostname"]), None)
else:
    cluster = clusters[0]

compute = Compute(cluster=cluster, queue=QUEUE_NAME, ppn=PPN)
print(f"Using cluster: {compute.cluster.hostname}, queue: {QUEUE_NAME}, ppn: {PPN}")

## 6. Create the Defect Formation Energy job
### 6.1. Create the multi-material job (defective + pristine)

In [ ]:
from mat3ra.notebooks_utils.api.job import wait_for_jobs_to_finish_async
from mat3ra.notebooks_utils.job import create_job
from mat3ra.notebooks_utils.ui import display_JSON
from mat3ra.utils.namespace import dict_to_namespace_recursive

defect_job_name = f"{MY_WORKFLOW_NAME} {saved_defective.formula} {timestamp}"
defect_job_response = create_job(
    api_client=client,
    materials=materials,
    workflow=defect_workflow,
    project_id=project_id,
    owner_id=ACCOUNT_ID,
    prefix=defect_job_name,
    compute=compute.to_dict(),
)

defect_job = dict_to_namespace_recursive(defect_job_response)
defect_job_id = defect_job._id
print(f"✅ Defect Formation Energy job created: {defect_job_id}")
display_JSON(defect_job_response)

### 6.2. Submit the Defect Formation Energy job and monitor the status

In [ ]:
client.jobs.submit(defect_job_id)
print(f"✅ Job {defect_job_id} submitted successfully!")

In [ ]:
await wait_for_jobs_to_finish_async(client.jobs, [defect_job_id], poll_interval=POLL_INTERVAL)

## 7. Retrieve results
### 7.1. Retrieve and visualize defect formation energy

In [ ]:
from mat3ra.notebooks_utils.ipython.entity.property.visualize import visualize_properties

defect_energy_data = client.properties.get_for_job(defect_job_id)
visualize_properties(defect_energy_data, title="Defect Formation Energy")